# Flight Cost Prediction — Machine Learning Project

**Purpose:** estimate a flight fare in Indian rupees from the route, travel date, airline, departure time, number of stops, cabin class, and how early the ticket is booked.

> **Important:** `flight_price_sample.csv` is a **synthetic teaching dataset**, not scraped booking data. The model demonstrates an end-to-end ML workflow and should not be used as a real quotation. For a real-world model, replace the sample CSV with a legally obtained, representative historical flight-price dataset and retrain.

## How to use
1. Install the packages in `requirements.txt`.
2. Run this notebook from the project folder.
3. Execute cells from top to bottom. The final cell starts a local API at `http://127.0.0.1:5000`.
4. Open `index.html` in a browser and submit the form.
5. Keep this notebook's kernel running while using the web page.

## 1. Import libraries and load the sample dataset

In [ ]:
from pathlib import Path
import math
import threading
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

# Run the notebook from the folder containing this notebook and CSV.
DATA_PATH = Path.cwd() / "flight_price_sample.csv"
if not DATA_PATH.exists():
    # Helps when Jupyter was launched from the parent folder.
    candidate = Path.cwd() / "flight_cost_prediction_project" / "flight_price_sample.csv"
    if candidate.exists():
        DATA_PATH = candidate
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "flight_price_sample.csv was not found. Put the CSV in the same folder as this notebook."
    )

raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns")
display(raw_df.head())

## 2. Explore and prepare the data

In [ ]:
print("Dataset shape:", raw_df.shape)
print("Missing values by column:")
display(raw_df.isna().sum().to_frame("missing_values"))
print("Fare summary (INR):")
display(raw_df["price_inr"].describe().to_frame())

# Feature engineering: extract calendar and departure-hour features.
df_model = raw_df.copy()
df_model["journey_date"] = pd.to_datetime(df_model["journey_date"], errors="coerce")
df_model["month"] = df_model["journey_date"].dt.month
df_model["day_of_week"] = df_model["journey_date"].dt.dayofweek
df_model["departure_hour"] = pd.to_datetime(
    df_model["departure_time"], format="%H:%M", errors="coerce"
).dt.hour

FEATURES = [
    "airline", "source", "destination", "travel_class",
    "month", "day_of_week", "departure_hour", "total_stops",
    "booking_days_before", "distance_km", "duration_mins"
]
TARGET = "price_inr"
df_model = df_model.dropna(subset=FEATURES + [TARGET]).copy()
X = df_model[FEATURES]
y = df_model[TARGET]

categorical_features = ["airline", "source", "destination", "travel_class"]
numeric_features = [c for c in FEATURES if c not in categorical_features]

print("Rows after cleaning:", len(df_model))
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

## 3. Train the model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=220, min_samples_leaf=2, random_state=42, n_jobs=-1
    )),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = {
    "MAE (INR)": mean_absolute_error(y_test, y_pred),
    "RMSE (INR)": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "R² score": r2_score(y_test, y_pred),
}
display(pd.DataFrame([metrics]).round(3))
print("Note: these metrics measure performance on synthetic sample data only.")

# Save the fitted pipeline so it can be reused later if desired.
MODEL_PATH = Path.cwd() / "flight_price_model.joblib"
joblib.dump(model, MODEL_PATH)
print("Saved model to:", MODEL_PATH.resolve())

## 4. Visualize actual vs predicted fares

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.45)
lims = [min(float(y_test.min()), float(np.min(y_pred))), max(float(y_test.max()), float(np.max(y_pred)))]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual fare (INR)")
plt.ylabel("Predicted fare (INR)")
plt.title("Actual vs Predicted Flight Fares — Synthetic Dataset")
plt.tight_layout()
plt.show()

## 5. Try one prediction inside the notebook

In [ ]:
# This example uses the same feature format as the web form.
CITY_COORDS = {
    "Delhi": (28.5562, 77.1000), "Mumbai": (19.0896, 72.8656),
    "Bengaluru": (13.1986, 77.7066), "Hyderabad": (17.2403, 78.4294),
    "Chennai": (12.9941, 80.1709), "Kolkata": (22.6547, 88.4467),
    "Jaipur": (26.8242, 75.8122), "Lucknow": (26.7606, 80.8893),
    "Ahmedabad": (23.0772, 72.6347), "Pune": (18.5822, 73.9197),
}

def haversine_km(source, destination):
    lat1, lon1 = CITY_COORDS[source]
    lat2, lon2 = CITY_COORDS[destination]
    radius = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return int(round(2 * radius * math.asin(math.sqrt(a))))

def build_features(airline, source, destination, journey_date, departure_time,
                   total_stops, travel_class, booking_days_before):
    travel_day = pd.to_datetime(journey_date)
    departure_hour = int(str(departure_time).split(":")[0])
    distance = haversine_km(source, destination)
    duration = int(round(42 + distance / 11.5 + int(total_stops) * 65))
    return pd.DataFrame([{
        "airline": airline, "source": source, "destination": destination,
        "travel_class": travel_class, "month": int(travel_day.month),
        "day_of_week": int(travel_day.dayofweek), "departure_hour": departure_hour,
        "total_stops": int(total_stops), "booking_days_before": int(booking_days_before),
        "distance_km": distance, "duration_mins": duration,
    }])[FEATURES]

example = build_features(
    "IndiGo", "Delhi", "Mumbai", "2026-12-15", "09:00", 0, "Economy", 21
)
example_prediction = float(model.predict(example)[0])
print(f"Example estimated fare: ₹{example_prediction:,.0f}")
display(example)

## 6. Start the API for the HTML frontend

Run this last cell after the training cells. It defines the prediction API **inside the notebook**—there is no separate Python server file. Leave the notebook kernel running, then open `index.html`.

In [ ]:
from flask import Flask, jsonify, request
from flask_cors import CORS
from werkzeug.serving import make_server

app = Flask("flight_cost_prediction_api")
CORS(app, resources={r"/*": {"origins": "*"}})

ALLOWED_CITIES = set(CITY_COORDS)
ALLOWED_AIRLINES = set(raw_df["airline"].dropna().unique())
ALLOWED_CLASSES = {"Economy", "Business"}

@app.get("/health")
def health():
    return jsonify({"status": "ok", "message": "Flight prediction API is running"})

@app.post("/predict")
def predict_fare():
    payload = request.get_json(silent=True) or {}
    required = ["airline", "source", "destination", "journey_date", "departure_time",
                "total_stops", "travel_class", "booking_days_before"]
    missing = [key for key in required if payload.get(key) in (None, "")]
    if missing:
        return jsonify({"error": "Missing required fields: " + ", ".join(missing)}), 400

    try:
        airline = str(payload["airline"])
        source = str(payload["source"])
        destination = str(payload["destination"])
        travel_class = str(payload["travel_class"])
        stops = int(payload["total_stops"])
        booking_days = int(payload["booking_days_before"])
        travel_date = pd.to_datetime(payload["journey_date"], errors="raise")
        time_text = str(payload["departure_time"])
        pd.to_datetime(time_text, format="%H:%M", errors="raise")

        if source not in ALLOWED_CITIES or destination not in ALLOWED_CITIES:
            raise ValueError("Choose a supported source and destination city.")
        if source == destination:
            raise ValueError("Source and destination must be different.")
        if airline not in ALLOWED_AIRLINES:
            raise ValueError("Choose a supported airline.")
        if travel_class not in ALLOWED_CLASSES:
            raise ValueError("Choose Economy or Business class.")
        if stops not in (0, 1, 2):
            raise ValueError("Stops must be 0, 1, or 2.")
        if not 0 <= booking_days <= 90:
            raise ValueError("Advance booking must be between 0 and 90 days.")
        if travel_date.date() < pd.Timestamp.today().date():
            raise ValueError("Travel date must be today or a future date.")

        features = build_features(
            airline, source, destination, travel_date.strftime("%Y-%m-%d"),
            time_text, stops, travel_class, booking_days
        )
        predicted_price = max(0, float(model.predict(features)[0]))
        distance = int(features.iloc[0]["distance_km"])
        duration = int(features.iloc[0]["duration_mins"])
        return jsonify({
            "predicted_price_inr": round(predicted_price),
            "route_distance_km": distance,
            "estimated_duration_minutes": duration,
            "model": "Random Forest Regressor",
            "disclaimer": "Educational estimate from synthetic sample data; not a live fare quotation."
        })
    except (ValueError, TypeError, KeyError) as exc:
        return jsonify({"error": str(exc)}), 400

# If this cell is executed a second time, stop the previous notebook server first.
try:
    server.shutdown()
except NameError:
    pass

server = make_server("127.0.0.1", 5000, app, threaded=True)
api_thread = threading.Thread(target=server.serve_forever, daemon=True)
api_thread.start()
print("✅ API running at http://127.0.0.1:5000")
print("Keep this notebook kernel running. Open index.html in your browser to predict fares.")

## 7. Project notes and future improvements
- This is a **regression** task because the output is a continuous number (fare in INR).
- The pipeline converts categorical fields with one-hot encoding and uses a Random Forest Regressor.
- Current route distances and durations are approximate, and the training fares are synthetic.
- For a production-grade project, use verified historical flight fare records, include booking timestamp and airline-specific details, monitor prediction errors, and retrain when prices shift.
- Stop the local API by interrupting the notebook kernel when finished.